# Lab 1: Judoor and HPC Access

**Course:** Machine Learning for Earth Observation with Supercomputers
**Duration:** 2 hours of hands-on work, plus 1-2 days of lead time for account approval
**You will work in two places:** on your own laptop, and on the JURECA supercomputer

By the end of this lab you can log in to JURECA, you know where your files should live, and you have
submitted one real job to the scheduler.

---

## How to use this notebook

Every command block is labelled with the machine it belongs to. Read the label before you type.

| Label | Meaning |
| --- | --- |
| **Run on your laptop** | Your own computer. PowerShell on Windows, Terminal on macOS or Linux. |
| **Run on JURECA** | After you have logged in. Bash on a JURECA login node. |

Other conventions:

- `UPPERCASE_IN_UNDERSCORES` means *replace this with your own value*. Never type the angle brackets
  or braces literally.
- Lines starting with `#` inside a command block are comments for you, not commands to type.
- Each section ends with a **Checkpoint**. Do not move on until your checkpoint passes. If you get
  stuck, jump to [Troubleshooting](#section-9-troubleshooting) and then ask on Slack.

> **Navigating with a screen reader or keyboard:** headings are in strict order and every table has a
> header row. Core work is marked with the word "Required"; optional work is marked with the word
> "Optional". Nothing in this lab relies on colour or an icon alone to carry meaning.

---

## Time budget

| Section | What you do | Where | Minutes |
| --- | --- | --- | --- |
| 1 | Understand what you are connecting to | read only | 5 |
| 2 | Create your Judoor account, join the course project | laptop, browser | 15 |
| 3 | Create an SSH key pair and register the public half | laptop, then browser | 25 |
| 4 | Log in for the first time | laptop | 10 |
| 5 | Learn where your files live and set the active project | JURECA | 15 |
| 6 | Build your course workspace | JURECA | 10 |
| 7 | Submit and inspect one real job | JURECA | 20 |
| 8 | Wrap up, checklist, troubleshooting | read only | 5 |

Sections 1 to 7 are **Required**. Everything after the "Optional extensions" divider is
**Optional**, for anyone who finishes early.


## Learning outcomes

When you finish, you will be able to:

| Priority | Outcome |
| --- | --- |
| Required | Create a Judoor account and join the `training2600` course project. |
| Required | Generate an SSH key pair that meets JSC security rules and register the public key. |
| Required | Log in to JURECA from Windows, macOS, or Linux without a password. |
| Required | Explain the difference between `$HOME`, `$PROJECT`, and `$SCRATCH`, and choose the right one. |
| Required | Set an active project with `jutil` and create a tidy course workspace. |
| Required | Submit a batch job, find its output, and confirm it ran. |
| Optional | Move files between your laptop and JURECA with `scp` or `rsync`. |
| Optional | Work on JURECA from VS Code with Remote SSH. |

### Prerequisites

- An email address you control (a university address is best).
- A GitHub account (you will need it in Lab 2).
- Basic terminal confidence: `cd`, `ls`, `mkdir`. If those are new to you, read the
  [Linux command line basics](https://ubuntu.com/tutorials/command-line-for-beginners) guide
  before the session.
- No prior HPC experience is assumed.

### A short glossary

The rest of the lab uses these words a lot.

| Term | Plain meaning |
| --- | --- |
| HPC | High Performance Computing. A big pile of computers you borrow through a queue instead of your laptop. |
| JSC | Juelich Supercomputing Centre, in Germany. It owns the machines this course uses. |
| JURECA | The specific supercomputer you will use. |
| Judoor | The website where you manage your JSC account, projects, and SSH keys. |
| Login node | A shared machine you land on after `ssh`. Type-light work only: editing, submitting jobs. Never train there. |
| Compute node | The heavy machines. You never log in directly; the scheduler puts your job there. |
| Node | One physical server. On JURECA a node has 128 CPU cores, and GPU nodes add 4 NVIDIA A100 GPUs. |
| SLURM | The scheduler. It decides who gets which nodes and when. |
| Partition | A named group of nodes with its own rules and time limits, like `dc-cpu` or `dc-gpu`. |
| Wall clock time | Real elapsed time your job holds the nodes. This is what you are billed for. |
| Batch job | A script you hand to SLURM to run later, unattended. |


## Section 1: What you are connecting to, and why

### The three pieces

| Piece | What it is | What you touch |
| --- | --- | --- |
| Judoor | Account and access portal for all JSC systems. | A browser. Register, join projects, upload SSH keys. |
| JURECA | The supercomputer. CPU nodes and A100 GPU nodes. | An SSH terminal, once you have a key registered. |
| SLURM | The scheduler inside JURECA. | Commands like `sbatch`, `squeue`, `scancel`. |

You cannot reach JURECA until Judoor has approved you and knows your public key. That ordering causes
most of the confusion on day one, so do the steps in order.

### Why a supercomputer for this course

- **Data volume.** Sentinel-2 scenes for a land-cover study reach hundreds of gigabytes easily.
  Downloading and tiling them on a laptop takes days.
- **GPUs.** Fine-tuning a geospatial foundation model needs an A100. Most laptops have neither the
  card nor the memory.
- **Parallel preprocessing.** Patch extraction is embarrassingly parallel across CPU cores.
- **Shared storage.** One copy of the dataset, visible to every node and to every teammate.

### What the course gives you

| Item | Value |
| --- | --- |
| Project (storage and membership) | `training2600` |
| Machine | JURECA, DC module |
| CPU partition | `dc-cpu`, 128 cores and 512 GiB per node, whole nodes only |
| GPU partition | `dc-gpu`, same nodes plus 4x A100 (40 GiB each) per node |
| Development partitions | `dc-cpu-devel` and `dc-gpu-devel`, max 2 hours, and they do have internet |

> **Two facts that trip up almost every new JSC user.** First, compute nodes are allocated whole. The
> smallest job you can submit takes an entire node, all 128 cores, and you are charged for all of them
> for as long as you hold them. Second, compute nodes on `dc-cpu` and `dc-gpu` have **no internet
> access**. A `pip install` inside such a job fails. Install on a login node or in a `devel` partition
> first, then run offline.

**Checkpoint 1:** you can say, out loud, which of Judoor, JURECA, and SLURM you log in to, which one
runs your code, and which one decides the queue order.


## Section 2: Create your Judoor account

Do this section first, today. Approval is usually quick, but it is not instant, and nothing else in
this lab works without it.

### Step 2.1 Register

**Run on your laptop** (browser):

1. Open <https://judoor.fz-juelich.de/register>.
2. Use your university email if you have one. JSC matches people to institutions on the email domain.
3. Pick a strong password you have not used elsewhere. A password manager makes this painless.
4. Complete the verification link in the email that arrives.

### Step 2.2 Turn on multi-factor authentication

Judoor protects account changes, including SSH key uploads, with a second factor. Set it up now, while
you are not on a clock in a two-hour lab.

**Run on your laptop** (browser):

1. In Judoor, open your account settings and find the multi-factor or TOTP option.
2. Scan the QR code with an authenticator app (FreeOTP, Authy, 1Password, or Google Authenticator).
3. Store the recovery codes somewhere safe. Without them, losing your phone means contacting support.

### Step 2.3 Join the course project

**Run on your laptop** (browser), while logged in:

1. Open <https://judoor.fz-juelich.de/projects/join/training2600>.
2. Read the usage policy. It is a real agreement about acceptable use and data handling.
3. Accept and submit.

Wait until the project shows in your project list before continuing.

### If it goes wrong

| Symptom | What to do |
| --- | --- |
| No verification email | Check spam and any university quarantine notice. Wait 10 minutes, then resend. |
| Join link does nothing | You are not logged in yet. Log in, then reopen the link. |
| Project access denied or pending for over a day | Post your Judoor username on the course Slack and ping the instructor. |
| Locked out after bad passwords | JSC temporarily blocks the source IP after repeated failures. Wait 15 minutes and retry slowly. |

**Checkpoint 2:** you are logged in to Judoor, MFA is on, and `training2600` appears in your project
list.


## Section 3: SSH keys

### Why keys, and why a passphrase is not optional

JSC does not accept password logins at all. You authenticate with a key pair: a **private** key that
never leaves your laptop, and a **public** key you upload to Judoor.

Two JSC rules matter here, and both are places where students lose time:

1. **Your private key must be protected by a passphrase.** A key without one is rejected. When
   `ssh-keygen` asks for a passphrase, you must type one. This is not the "just press Enter" default
   you may have seen in other tutorials.
2. **You must declare where you connect from.** When you upload a public key, Judoor asks for the
   hosts or subnets allowed to use it. Your key is ignored if that range does not cover you.

To avoid typing the passphrase for every connection, you will use a **ssh-agent** in Step 3.4. That
gives you password-free logins while the key itself stays protected.

### Step 3.1 Generate the key pair

**Run on your laptop** (PowerShell on Windows, or Terminal on macOS and Linux):

```powershell
# Create the .ssh folder if it does not exist yet
mkdir $HOME\.ssh

# Generate an Ed25519 key. You WILL be asked for a passphrase: type a real one.
ssh-keygen -a 100 -t ed25519 -C "you@yourinstitution.edu" -f $HOME\.ssh\jsc_jureca
```

On macOS or Linux, use `~/.ssh` instead of `$HOME\.ssh`:

```bash
mkdir -p ~/.ssh
ssh-keygen -a 100 -t ed25519 -C "you@yourinstitution.edu" -f ~/.ssh/jsc_jureca
```

`ssh-keygen` writes two files. `jsc_jureca` is private. `jsc_jureca.pub` is public and is the only one
you ever share.

> **If `ssh-keygen` is missing on Windows:** install the OpenSSH client. Settings, then Apps, then
> Optional features, then Add a feature, then look for "OpenSSH Client". Reopen PowerShell afterwards.

### Step 3.2 Copy the public key

**Run on your laptop:**

```powershell
# Windows PowerShell: prints the key and also puts it on the clipboard
Get-Content $HOME\.ssh\jsc_jureca.pub
Get-Content $HOME\.ssh\jsc_jureca.pub | Set-Clipboard
```

```bash
# macOS: prints and copies in one step
cat ~/.ssh/jsc_jureca.pub | pbcopy
```

```bash
# Linux: xclip or wl-clipboard must be installed
cat ~/.ssh/jsc_jureca.pub
xclip -selection clipboard < ~/.ssh/jsc_jureca.pub
```

The key is one long line starting with `ssh-ed25519`. Copy it whole, with no line breaks added.

### Step 3.3 Upload it to Judoor

**Run on your laptop** (browser):

1. Open Judoor and go to the **JURECA Manage SSH-Keys** section. You will be asked for your second
   factor.
2. Paste the public key into the field for the key and options string.
3. Add a source restriction so the key is actually usable. Judoor suggests a range for you. If you are
   on the University of Iceland network, use:

   ```text
   from="130.208.0.0/16"
   ```

   Owner of that range: Rannsokna- og Haaskolanelid Islands hf (Rannis, the Icelandic university
   network).
4. Save. Give the key a recognisable name or comment so you can revoke the right one later.

> **Working from home or a cafe?** Your home IP changes, and a `from=` restriction that does not match
> it means login fails with `Permission denied (publickey)`. Two clean options: register a second key
> restricted to your current network each time you switch locations, or use the university VPN and
> register the VPN range. Avoid `from="0.0.0.0/0"`, which lets anyone who ever copies your key log in
> as you.

### Step 3.4 Let the agent remember your passphrase

**Run on your laptop** (PowerShell):

```powershell
# Start the agent service once, then set it to start automatically
Get-Service ssh-agent | Set-Service -StartupType Automatic
Start-Service ssh-agent

# Add your key. You type the passphrase one final time per boot.
ssh-add $HOME\.ssh\jsc_jureca
```

```bash
# macOS and Linux
eval "$(ssh-agent -s)"
ssh-add ~/.ssh/jsc_jureca
```

On macOS you can store the passphrase in the keychain so it survives reboots:

```bash
ssh-add --apple-use-keychain ~/.ssh/jsc_jureca
```

**Checkpoint 3:** the key pair exists, the public half is in Judoor with a source range that covers
where you are sitting, and `ssh-add -l` lists your key.


## Section 4: First login

### Step 4.1 Write an SSH config file

A config file means you type `ssh jureca` instead of a long command, and it guarantees the right key is
offered.

**Run on your laptop:** open `$HOME/.ssh/config` in a text editor, or create it:

```powershell
notepad $HOME\.ssh\config      # Windows
code $HOME\.ssh\config         # any OS, if you use VS Code
```

Put this in it, then replace `YOUR_JUDOOR_USERNAME` with your actual Judoor username:

```text
Host jureca
    HostName jureca.fz-juelich.de
    User YOUR_JUDOOR_USERNAME
    IdentityFile ~/.ssh/jsc_jureca
    IdentitiesOnly yes
    ServerAliveInterval 60
    ServerAliveCountMax 10
```

`IdentitiesOnly yes` stops your agent from offering every other key you own, which otherwise gets you
banned by the firewall for "too many authentication attempts".

### Step 4.2 Connect

**Run on your laptop:**

```bash
ssh jureca
```

The first time, SSH asks you to confirm the server fingerprint. Type `yes`. You should land at a bash
prompt on a login node, with a message of the day above it.

```bash
# Confirm who and where you are
whoami
hostname
```

### Step 4.3 Know the login node rules

Login nodes are shared by everyone in the course and beyond. They are for editing files, submitting
jobs, and light checks.

| Do on a login node | Never do on a login node |
| --- | --- |
| Edit scripts, run `git` | Train models |
| Submit and monitor jobs | Large data conversions or unzipping gigabytes |
| Install packages into your own environment | Long-running loops, `watch`, or `sleep` marathons |

If you need compute, ask SLURM for it. That is what Section 7 teaches.

**Checkpoint 4:** `ssh jureca` drops you on a login node prompt with no password prompt and no
`Permission denied` message.


## Section 5: Where your files live

### The three storage areas

Everything on JURECA lives on one parallel filesystem (GPFS) that every node and every login node can
see. Which directory you use is a real decision, not a formality.

| Variable | Backed up? | Auto-deleted? | Typical quota | Use it for |
| --- | --- | --- | --- | --- |
| `$HOME` | Yes | No | 20 GB soft, 22 GB hard | Dotfiles, SSH config, small configs. Not data. |
| `$PROJECT` | Yes | No | 15 TB soft per project | Code, environments, scripts, small results, anything you would hate to lose. |
| `$SCRATCH` | **No** | Files after 90 days, empty dirs after 3 days | 90 TB soft per project | Big intermediates, patches, anything regenerable. |

> **Read that "No" twice.** `$SCRATCH` has no backup and files vanish 90 days after they were last
> touched. Never leave a trained checkpoint or a cleaned dataset only in `$SCRATCH`. Copy anything
> worth keeping into `$PROJECT`.

### Step 5.1 Set your active project

You belong to several projects, so JSC names the variables per project, like `$PROJECT_training2600`.
The suffix-less `$PROJECT` and `$SCRATCH` only exist once you activate a project.

**Run on JURECA:**

```bash
# Which projects do I belong to?
jutil user project

# Make the course project active for this shell session
jutil env activate -p training2600

# Confirm it worked
echo "PROJECT = $PROJECT"
echo "SCRATCH = $SCRATCH"
echo "HOME    = $HOME"
```

The project-specific variables are always available, with or without activation:

```bash
echo $PROJECT_training2600
echo $SCRATCH_training2600
```

> **Add it to your shell startup so you stop forgetting.** Append the activate line to `~/.bashrc` and
> every new login node session, and every job script that sources your profile, starts with the course
> project active:
>
> ```bash
> echo 'jutil env activate -p training2600' >> ~/.bashrc
> ```

### Step 5.2 Look around

**Run on JURECA:**

```bash
# Your home directory, and how full it is
cd "$HOME"
pwd
jutil user dataquota

# The course project directory, shared with everyone in training2600
cd "$PROJECT"
pwd
ls -la
```

`jutil user dataquota` prints your usage against the soft and hard limits. Run it weekly. Going over
the file-count limit, not just the size limit, is the usual surprise, and it makes a directory
unwritable in a confusing way.

**Checkpoint 5:** `jutil env activate -p training2600` runs cleanly, and `echo $PROJECT` prints a real
path rather than nothing.


## Section 6: Build your course workspace

A predictable layout pays for itself by Lab 4. Everyone in the course uses the same shape, which also
makes it easy for instructors to help you, because your paths look like theirs.

**Run on JURECA:**

```bash
# Go to the course project directory
cd "$PROJECT"

# One directory per person, named after your Judoor username
mkdir -p "$USER"
cd "$USER"

# The standard course subdirectories
mkdir -p repos envs scripts logs data results

# Confirm. `tree` is not installed here, so use find.
find . -maxdepth 1 -type d | sort
```

You should see:

```text
.
./data
./envs
./logs
./repos
./results
./scripts
```

### What goes where

| Directory | Contents |
| --- | --- |
| `repos` | Git clones, including the course repository you add in Lab 2 |
| `envs` | Virtual environments or conda environments |
| `scripts` | Your job scripts and small Python utilities |
| `logs` | Job stdout and stderr, once you point `sbatch` here |
| `data` | Datasets and patches. Move to `$SCRATCH` if they get large. |
| `results` | Checkpoints, metrics, figures. Things you must not lose. |

### Step 6.1 A first script

**Run on JURECA:**

```bash
cd "$PROJECT/$USER"

cat > scripts/hello_hpc.py << 'PYEOF'
#!/usr/bin/env python3
# Print a little information about the machine we landed on.
import os
import socket

print(f"Hello from HPC!")
print(f"Hostname : {socket.gethostname()}")
print(f"User     : {os.getenv('USER')}")
print(f"Project  : {os.getenv('PROJECT')}")
print(f"Job id   : {os.getenv('SLURM_JOB_ID', 'not inside a job')}")
PYEOF

python3 scripts/hello_hpc.py
```

The `SLURM_JOB_ID` line is the interesting one. Run now, it prints `not inside a job`. Run again inside
a job in Section 7, and it prints a number. That single difference is your proof that the scheduler ran
your code somewhere else.

> **That `cat > file << 'PYEOF'` trick is a heredoc, and it only works in bash.** If you try it in
> Windows PowerShell it fails with a parse error. On your laptop, create files with an editor instead.

**Checkpoint 6:** `find . -maxdepth 1 -type d` shows all six subdirectories, and
`python3 scripts/hello_hpc.py` prints your username and `not inside a job`.


## Section 7: Your first job

This is the part that makes it an HPC course. You will write a script that asks SLURM for a node, run
your Python file on it, and read the output back.

### Step 7.1 Write the job script

**Run on JURECA:**

```bash
cd "$PROJECT/$USER"

cat > scripts/lab1_hello.sh << 'EOF'
#!/bin/bash
#SBATCH --job-name=lab1-hello
#SBATCH --partition=dc-cpu-devel
#SBATCH --account=training2600
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --time=00:10:00
#SBATCH --output=logs/%x-%j.out
#SBATCH --error=logs/%x-%j.err

# sbatch starts your script in the directory you submitted from, but be explicit
# so the job behaves the same no matter where you typed sbatch.
cd "$SLURM_SUBMIT_DIR"

echo "Job started  : $(date)"
echo "Submitted from: $SLURM_SUBMIT_DIR"
echo "Node list    : $SLURM_JOB_NODELIST"
echo "CPUs on node : $SLURM_CPUS_ON_NODE"

srun python3 scripts/hello_hpc.py

echo "Job finished : $(date)"
EOF
```

What each line buys you:

| Directive | Effect |
| --- | --- |
| `--job-name=lab1-hello` | Names the job, and feeds `%x` in the output filename. |
| `--partition=dc-cpu-devel` | The development partition: short jobs, quick turnaround, and it has internet. |
| `--account=training2600` | Which project budget pays for the job. |
| `--nodes=1` | One whole node. This is the minimum on JURECA, not a small request. |
| `--ntasks=1` | One process. You will raise this when you learn MPI and DDP. |
| `--cpus-per-task=4` | Four cores for that process. |
| `--time=00:10:00` | Wall clock limit. SLURM kills the job at 10 minutes. Ask for little while learning. |
| `--output` / `--error` | Where stdout and stderr go. `%x` is the job name, `%j` the job id. |

> **Why `dc-cpu-devel` and not `dc-cpu`.** The `devel` partitions cap you at 2 hours and a few nodes,
> which is exactly right for a 10-minute test, and they return quickly. They also have internet access,
> which `dc-cpu` does not. Note that if you omit `--partition` entirely, JURECA's default is `dc-gpu`,
> which wastes GPU time on a CPU test. Always state the partition.

> **If Slurm rejects the account name,** the budget account may differ from the project name. Run
> `jutil user project`, and ask the instructor on Slack for the exact `--account` value for the course.

### Step 7.2 Submit it

**Run on JURECA:**

```bash
cd "$PROJECT/$USER"
sbatch scripts/lab1_hello.sh
```

You get one line back, which is the most useful line in this lab:

```text
Submitted batch job 12345678
```

Save that number. Everything below needs it.

```bash
# Put it in a variable so the next commands are copy-paste ready
JOBID=12345678
```

### Step 7.3 Watch it

**Run on JURECA:**

```bash
# Your own jobs, and why they are waiting if they are
squeue --me --format="%.10i %.14j %.2t %.10M %.12P %.6D %.20R"

# Everything queued on the CPU partition, to see how busy the machine is
squeue -p dc-cpu | head -20

# Full detail on one job, including the node it landed on
scontrol show job "$JOBID"
```

The `Reason` and `Dependency` fields in that output explain a job that will not start.

The state column tells you what is happening:

| State | Meaning | What to do |
| --- | --- | --- |
| `PD` | Pending. Waiting for a free node, or rejected by a limit. | Read the reason field. Check partition and account. |
| `CF` | Configuring. Nodes are being prepared. | Normal, lasts seconds. |
| `R` | Running. | Your code is on a compute node now. |
| `CG` | Completing. Cleanup in progress. | Normal, lasts seconds. |

### Step 7.4 Read the result

**Run on JURECA:**

```bash
# What the job printed
cat logs/lab1-hello-$JOBID.out

# Any errors. Check this even when the output looks fine.
cat logs/lab1-hello-$JOBID.err

# After the job ends: what actually happened, and how much it cost you
sacct -j "$JOBID" --format=JobID,JobName,Partition,State,ExitCode,Elapsed,AllocCPUS,MaxRSS
```

In `logs/lab1-hello-$JOBID.out` you should now see a real job id where Section 6 printed
`not inside a job`, and a hostname that is **not** the login node you typed the command on. That is
your proof.

### Step 7.5 Clean up

**Run on JURECA:**

```bash
# Stop a job that is running or stuck in the queue
scancel "$JOBID"

# Stop every job you own that is pending or running. Note there is no `scancel --me`:
# the filter is `--user`, and `--state` takes the long state names.
scancel --user="$USER" --state=PENDING,RUNNING

# Jobs you submitted today, including finished ones
sacct --starttime=today --format=JobID,JobName,State,Elapsed
```

`scancel` on a pending job is free. `scancel` on a running job still costs the core-hours you already
used, so kill long jobs as soon as you know they are wrong.

**Checkpoint 7:** you have a `logs/lab1-hello-<jobid>.out` file whose hostname differs from your login
node, and `sacct` shows the job with `State=COMPLETED` and `ExitCode=0`.


## Section 8: Wrap up

### What you now have

| Required outcome | How you prove it |
| --- | --- |
| Judoor account, MFA on, in `training2600` | Project visible in your Judoor project list |
| SSH key pair registered with a source range | `ssh-add -l` lists it, Judoor shows it |
| Working login | `ssh jureca` reaches a prompt with no password |
| Storage areas understood | `jutil env activate -p training2600` then `echo $PROJECT` |
| Course workspace created | `find "$PROJECT/$USER" -maxdepth 1 -type d` shows six dirs |
| One job run | `logs/lab1-hello-<jobid>.out` exists, `sacct` says `COMPLETED` |

### Habits that will save you later

1. Set the active project in `~/.bashrc`, so `$PROJECT` is never empty in a job script.
2. Ask for the smallest job that works. Whole nodes, and you pay for all 128 cores while you wait.
3. Always state `--partition`. The default is `dc-gpu`.
4. Check the `.err` file even when the `.out` file looks fine.
5. Keep checkpoints and cleaned data out of `$SCRATCH`. It is deleted after 90 days and never backed up.
6. Never put a private key on JURECA storage. It is against JSC policy and will get flagged.
7. No internet on `dc-cpu` and `dc-gpu`. Install packages on a login node or a `devel` partition.

### Coming in Lab 2

- Launching Jupyter-JSC, so notebooks run on JSC rather than your laptop.
- Git on JURECA. Note that JSC blocks **outgoing** SSH connections and disables agent forwarding, so
  `git clone` over an `git@github.com:...` SSH URL will not work there. You will use HTTPS with a
  personal access token instead.
- Creating a Python environment and registering it as a Jupyter kernel.

### Getting help

| Route | When |
| --- | --- |
| Course Slack | Anything course specific. Fastest route during the week. Include the exact command and the exact error text. |
| Lab instructor, s.hashim@fz-juelich.de | Account, access, and project membership problems. |
| JSC support, sc@fz-juelich.de | System faults, quota increases, anything about the machine itself. |
| JURECA documentation | <https://apps.fz-juelich.de/jsc/hps/jureca/> |
| Judoor portal | <https://judoor.fz-juelich.de> |

Asking well gets you an answer faster. Paste four things: the command you ran, the full error, the
partition and account you used, and what you expected instead.


## Section 9: Troubleshooting

Work down the table. The first two rows cover most of the questions asked in this lab.

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `Permission denied (publickey)` | Key not uploaded, wrong key offered, or your IP is outside the `from=` range | Check the key is in Judoor. Add `IdentitiesOnly yes`. Confirm your network matches the range. Run `ssh -vvv jureca` and read which keys are offered. |
| `Received disconnect ... too many bad auths` | The agent offered several keys and the firewall gave up | Set `IdentitiesOnly yes` in the host block, or `ssh-add -D` then add only the JSC key. |
| Passphrase demanded on every connect | The agent is not running or the key was never added | `Start-Service ssh-agent` on Windows, then `ssh-add`. |
| `ssh: Could not resolve hostname` | Typo in the config, or no network | `nslookup jureca.fz-juelich.de`. Check the `Host` name you typed matches the config. |
| `bash: cd: $PROJECT: No such file or directory` with an empty path | No active project in this shell | `jutil env activate -p training2600`, or use `$PROJECT_training2600`. |
| `sbatch: error: Batch job submission failed: Invalid account or account/partition combination` | Wrong `--account`, or a partition your project cannot use | `jutil user project`. Confirm the account and partition with the instructor. |
| Job sits in `PD` forever | Partition full, per-user job limit, or wall time above the partition maximum | `squeue --me` and read the reason. Try `dc-cpu-devel`. Lower `--time`. |
| Job finishes in seconds with nothing in `.out` | Script path wrong, or it died at once | Read the `.err` file. Confirm `cd "$SLURM_SUBMIT_DIR"` is present and paths are relative to the submit directory. |
| `pip install` inside a job fails or hangs | `dc-cpu` and `dc-gpu` have no internet | Install on a login node or in a `devel` partition, then run offline. |
| `tree: command not found` | Not installed on login nodes | `find . -maxdepth 1 -type d \| sort` |
| `module avail Python` shows almost nothing | JSC modules are hierarchical, so you must load a stage or compiler first | `module spider Python` to search, then `module load Stages/2024` and `module avail Python` |
| `Write error, Disk quota exceeded` | File count or size limit hit, often in `$HOME` | `jutil user dataquota`. Clear caches, especially pip and conda caches in `$HOME`. |
| Heredoc gives a syntax error | You ran a bash heredoc in PowerShell | Run it on JURECA, or create the file with an editor locally. |


---

# Optional extensions

Everything below is **Optional**. Return to it once every Required checkpoint passes, or after the
session. It is here because it is genuinely useful in later labs, not because you must finish it today.


## Optional A: Move files between your laptop and JURECA

Login nodes are the intended gateway for modest files. For large transfers, JSC prefers the dedicated
JUDAC gateway; you will not need it in this course.

**Run on your laptop.** These work because your `~/.ssh/config` defines the `jureca` host.

```bash
# Where your course workspace lives on JURECA. Type the real path; do not use $PROJECT here,
# because that variable belongs to your laptop shell and is almost certainly empty.
DEST=jureca:/p/project/training2600/YOUR_USER/

# Upload one file into your scripts directory
scp scripts/lab1_hello.sh "$DEST"scripts/

# Upload a whole directory
scp -r myproject/ "$DEST"

# Download one file into your current laptop directory
scp jureca:/p/project/training2600/YOUR_USER/results/metrics.csv ./

# Best for repeated or large copies: sends only what changed, and resumes
rsync -avzP myproject/ "$DEST"
```

The `$DEST` variable is set on your laptop, so it expands there, which is what you want. The trap is
writing `$PROJECT` in a `scp` destination: `$PROJECT` is only defined in a JURECA shell, so on your
laptop it expands to nothing and the file lands somewhere surprising.

| Flag | Meaning |
| --- | --- |
| `-a` | Archive mode: keeps permissions, times, and symlinks |
| `-v` | Verbose, so you can see progress |
| `-z` | Compress in flight. Helps text, does not help already-compressed imagery. |
| `-P` | Show progress and keep partial files so you can resume |
| `-r` | Recursive, for directories, with `scp` only |

> **Windows note:** `rsync` is not installed in PowerShell by default. Use `scp`, which is part of the
> OpenSSH client, or run `rsync` inside WSL or Git Bash. Also, `DEST=...` is bash syntax. In
> PowerShell, write `$DEST = "jureca:/p/project/training2600/YOUR_USER/"` instead, or skip the
> variable and type the full destination each time.

**Sanity check:** after any upload, run `ls -la` in the destination on JURECA and confirm the file size
is not zero.


## Optional B: Edit JURECA files from VS Code

Remote SSH gives you a real editor on the cluster, with the terminal already logged in.

1. In VS Code, install the **Remote - SSH** extension from Microsoft.
2. Press `Ctrl+Shift+P` (`Cmd+Shift+P` on macOS), type **Remote-SSH: Connect to Host**, and pick
   `jureca`. It reads the config file you wrote in Section 4.
3. Accept the fingerprint prompt, then **Open Folder** and choose your course workspace.

Once connected:

- The integrated terminal is already on a JURECA login node. No separate `ssh` needed.
- Install the Python extension **on the remote**, when the badge says it will install in the remote.
- Git runs on the cluster, which is what you want for Lab 2.

> **Keep it light.** The remote extension host runs on a login node and consumes memory. Close
> notebooks and language servers you are not using, and never run training from the VS Code terminal
> directly. Submit a job instead.

Useful settings, added to your user `settings.json`:

```json
{
  "remote.SSH.showLoginTerminal": true,
  "files.autoSave": "afterDelay",
  "files.autoSaveDelay": 2000
}
```

`showLoginTerminal` matters, because when a connection fails the log tells you why.


## Optional C: Interactive sessions, and a GPU check

An interactive allocation is a middle ground between a login node and a batch job: you get real compute
nodes, but you type commands and see results immediately.

**Run on JURECA:**

```bash
# Ask for one CPU node for 30 minutes, then get a shell on it
salloc --partition=dc-cpu-devel --account=training2600 --nodes=1 --time=00:30:00

# You are now on a compute node. Confirm it.
hostname
echo "$SLURM_JOB_ID"

# Run one task on it
srun python3 scripts/hello_hpc.py

# Give the node back
exit
```

For a GPU session, request the GPU development partition and one card. Without `--gres` you are given
all four GPUs on the node, which wastes course resources.

```bash
salloc --partition=dc-gpu-devel --account=training2600 --nodes=1 \
       --gres=gpu:1 --time=00:30:00

# Inside the allocation
module load Stages/2024
module avail CUDA
nvidia-smi
```

`nvidia-smi` should list one visible A100. If it lists four, your `--gres` request did not apply.

> **Do not leave an interactive allocation open while you read or eat.** The node is yours and the
> meter is running. `exit` when you are done, or set a short `--time` so SLURM ends it for you.


## Optional D: Software modules and your first Python environment

JSC does not put every program on your `PATH`. It uses *modules*, and loading one pulls in its
dependencies. The module tree is hierarchical, so `module avail` shows a short list until you load a
compiler or a stage first.

**Run on JURECA:**

```bash
# Search everything, regardless of what is loaded
module spider Python
module spider PyTorch

# Load a software stack, then look again
module load Stages/2024
module avail Python
module load Python
python3 --version
module list
```

`module spider` is the command to remember. If you think JSC has it, `module spider <name>` answers
yes or no.

Then build your own environment, which is what Lab 2 will actually train in:

```bash
cd "$PROJECT/$USER"
module load Python
python3 -m venv envs/lab1
source envs/lab1/bin/activate
which python
pip install --upgrade pip
pip install numpy
python -c "import numpy; print(numpy.__version__)"
deactivate
```

> **Never `pip install --user` into `$HOME` for course work, and never install into the system Python.**
> It bloats your small home quota and breaks when a module version changes. Use a virtual environment
> inside `$PROJECT/$USER/envs`, as above.

Record what you loaded. When a script works on one day and not the next, the module list is usually why:

```bash
module list > "$PROJECT/$USER/logs/modules-$(date +%F).txt"
```


## Lab 1 completion checklist

Tick a box only when you have actually run the check in the right-hand column.

### Required

- [ ] Judoor account created, verified, and MFA enabled
- [ ] `training2600` listed in my Judoor projects
- [ ] Ed25519 key pair generated **with a passphrase**
- [ ] Public key uploaded to Judoor with a source range covering my current network
- [ ] `~/.ssh/config` has a `jureca` host with `IdentitiesOnly yes`
- [ ] `ssh jureca` reaches a login node prompt with no password prompt
- [ ] `jutil env activate -p training2600` succeeds and `echo $PROJECT` prints a path
- [ ] I can explain, in one sentence each, when to use `$HOME`, `$PROJECT`, and `$SCRATCH`
- [ ] I know that `$SCRATCH` is not backed up and purges after 90 days
- [ ] `$PROJECT/$USER` contains `repos envs scripts logs data results`
- [ ] `scripts/hello_hpc.py` runs and prints `not inside a job`
- [ ] `sbatch scripts/lab1_hello.sh` returned a job id
- [ ] `logs/lab1-hello-<jobid>.out` shows a hostname that is not the login node
- [ ] `sacct -j <jobid>` shows `State=COMPLETED` and `ExitCode=0`
- [ ] I ran `scancel` on at least one job, pending or running

### Optional

- [ ] Uploaded and downloaded a file with `scp`, and verified the size on the far side
- [ ] Used `rsync -avzP` for a directory copy
- [ ] Connected to JURECA with VS Code Remote SSH and opened my workspace
- [ ] Held an interactive `salloc` allocation and ran `srun` inside it
- [ ] Saw one A100 with `nvidia-smi` in a `dc-gpu-devel` allocation
- [ ] Created a virtual environment in `$PROJECT/$USER/envs` and installed a package into it

### What to hand in

Send your instructor, or keep for yourself, these three lines of evidence:

```bash
whoami
sacct --starttime=today --format=JobID,JobName,Partition,State,ExitCode,Elapsed | head -10
find "$PROJECT/$USER" -maxdepth 1 -type d | sort
```

You have HPC access now. Everything from Lab 2 onward builds on it, so if any Required box is still
unticked, fix it before the next session rather than during it.
